# MediFlow Hair — 공개 데이터 후보 v1 패키징

학습을 다시 하지 않고 완료된 실험의 선택 모델을 검증하여 별도 Drive 폴더와 ZIP으로 보관합니다. 기존 실험 파일은 수정하지 않습니다.

- 선택 모델: EfficientNet-B1, 256×256, Label Smoothing 0.05, Stage 2 총 15 Epoch
- 선택 기준: Validation Accuracy
- 상태: 실제 USB 현미경 데이터로 검증하기 전의 공개 데이터 후보 v1
- 함께 기록할 대안: B0 Cross Entropy, B0 Label Smoothing 0.05


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib
import json
import shutil
from datetime import datetime
from pathlib import Path

import keras
import numpy as np
import pandas as pd

print('Keras:', keras.__version__)


## 1. 완료된 실험 비교

Test 수치는 선택 후 설명 자료이며 선택에는 사용하지 않습니다.


In [ ]:
MY_DRIVE = Path('/content/drive/MyDrive')
CLASS_NAMES = ['모낭사이홍반', '미세각질', '비듬', '탈모', '피지과다']
DATA_SHA256 = '2ac7260663cf69835ba50edb6ae8c7e7ac13be9c73b9f7ea24f60e0342e7e156'

CE_B0_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_two_stage_20260907_064214'
LS_B0_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_label_smoothing_005_20260907_103757'
FOCAL_B0_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_focal_gamma_15_20260907_105525'
B1_10_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_efficientnetb1_ls005_20260907_111711'
B1_15_ROOT = MY_DRIVE / 'mediflow_experiments' / 'hair' / 'clean_256_efficientnetb1_ls005_extended5_20260907_114654'

experiments = pd.DataFrame([
    {'experiment': 'B0 CE', 'validation_accuracy': 0.7699680328369141, 'test_accuracy': 0.7891373801916933, 'macro_f1': 0.7886477230068456, 'result_root': str(CE_B0_ROOT)},
    {'experiment': 'B0 LS0.05', 'validation_accuracy': 0.7739616632461548, 'test_accuracy': 0.786741214057508, 'macro_f1': 0.7863134458429594, 'result_root': str(LS_B0_ROOT)},
    {'experiment': 'B0 Focal1.5', 'validation_accuracy': 0.7675718665122986, 'test_accuracy': 0.7907348242811502, 'macro_f1': 0.7905122039602036, 'result_root': str(FOCAL_B0_ROOT)},
    {'experiment': 'B1 LS0.05 stage2-10', 'validation_accuracy': 0.7763578295707703, 'test_accuracy': 0.7787539936102237, 'macro_f1': 0.778812327205678, 'result_root': str(B1_10_ROOT)},
    {'experiment': 'B1 LS0.05 stage2-15', 'validation_accuracy': 0.7771565318107605, 'test_accuracy': 0.7883386581469649, 'macro_f1': 0.7885764577048346, 'result_root': str(B1_15_ROOT)},
])
selected_row = experiments.loc[experiments['validation_accuracy'].idxmax()]
if selected_row['experiment'] != 'B1 LS0.05 stage2-15':
    raise ValueError(f'예상과 다른 후보가 선택되었습니다: {selected_row.to_dict()}')
display(experiments)
print('Validation 기준 선택:', selected_row['experiment'])


## 2. 선택 모델 계약 검증

데이터 식별값, 클래스 순서, 입력·출력과 내부 Rescaling(1/255)을 확인합니다.


In [ ]:
SOURCE_MODEL_PATH = B1_15_ROOT / 'best_model.keras'
SOURCE_CONFIG_PATH = B1_15_ROOT / 'training_config.json'
for path in (SOURCE_MODEL_PATH, SOURCE_CONFIG_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'필수 파일이 없습니다: {path}')
source_config = json.loads(SOURCE_CONFIG_PATH.read_text(encoding='utf-8'))
if source_config.get('data_zip_sha256') != DATA_SHA256:
    raise ValueError('학습 데이터 식별값이 다릅니다.')
if source_config.get('class_names') != CLASS_NAMES:
    raise ValueError('클래스 순서가 다릅니다.')
if source_config.get('image_size') != [256, 256]:
    raise ValueError('입력 크기가 다릅니다.')
if source_config.get('selected_training') != 'b1_stage2_extended':
    raise ValueError('선택 학습 기록이 다릅니다.')

model = keras.models.load_model(SOURCE_MODEL_PATH, compile=False)
if tuple(model.input_shape[1:]) != (256, 256, 3) or int(model.output_shape[-1]) != 5:
    raise ValueError(f'모델 입출력이 다릅니다: {model.input_shape}, {model.output_shape}')
def flatten_layers(layer):
    result = []
    for child in getattr(layer, 'layers', []):
        result.append(child)
        result.extend(flatten_layers(child))
    return result
rescaling = [layer for layer in flatten_layers(model) if isinstance(layer, keras.layers.Rescaling)]
def is_internal_one_over_255(layer):
    scale = np.asarray(layer.scale, dtype=np.float64)
    return scale.size == 1 and np.isclose(scale.item(), 1 / 255)
if not any(is_internal_one_over_255(layer) for layer in rescaling):
    raise ValueError('내부 Rescaling(1/255)을 확인하지 못했습니다.')
dummy = model.predict(np.zeros((1, 256, 256, 3), dtype=np.float32), verbose=0)
if dummy.shape != (1, 5) or not np.isclose(dummy.sum(), 1.0, atol=1e-5):
    raise ValueError('Softmax 출력 계약이 다릅니다.')
print('선택 모델 계약 검증 완료:', model.count_params(), 'parameters')


## 3. 후보 패키지와 ZIP 생성


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
PACKAGE_ROOT = MY_DRIVE / 'mediflow_models' / 'hair' / f'public_candidate_v1_b1_256_ls005_{RUN_ID}'
if PACKAGE_ROOT.exists():
    raise FileExistsError(f'패키지가 이미 있습니다: {PACKAGE_ROOT}')
PACKAGE_ROOT.mkdir(parents=True)
MODEL_DESTINATION = PACKAGE_ROOT / 'hair_model.keras'
shutil.copy2(SOURCE_MODEL_PATH, MODEL_DESTINATION)
for artifact in ('training_config.json', 'classification_report.csv', 'confusion_matrix.png', 'b1_extension_curves.png'):
    source = B1_15_ROOT / artifact
    if not source.is_file():
        raise FileNotFoundError(f'평가 산출물이 없습니다: {source}')
    shutil.copy2(source, PACKAGE_ROOT / f'source_{artifact}')
model_sha256 = sha256_file(MODEL_DESTINATION)
experiments.to_csv(PACKAGE_ROOT / 'experiment_comparison.csv', index=False, encoding='utf-8-sig')
(PACKAGE_ROOT / 'class_names.json').write_text(json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2), encoding='utf-8')
preprocessing = {'input_shape': [256, 256, 3], 'color_order': 'RGB', 'input_dtype': 'float32', 'input_pixel_range': [0, 255], 'external_normalization': False, 'internal_rescaling': '1/255', 'output': 'softmax probabilities in class_names.json order'}
(PACKAGE_ROOT / 'preprocessing.json').write_text(json.dumps(preprocessing, ensure_ascii=False, indent=2), encoding='utf-8')
manifest = {
    'status': 'public_data_candidate_v1_not_device_validated',
    'created_at': datetime.now().isoformat(), 'source_result_root': str(B1_15_ROOT),
    'model_file': 'hair_model.keras', 'model_sha256': model_sha256,
    'model_size_bytes': MODEL_DESTINATION.stat().st_size, 'model_parameter_count': int(model.count_params()),
    'backbone': 'EfficientNet-B1', 'image_size': [256, 256],
    'loss': 'Categorical Crossentropy with Label Smoothing 0.05',
    'stage2_total_epochs': 15, 'selection_metric': 'validation_accuracy',
    'validation_accuracy': float(selected_row['validation_accuracy']), 'test_accuracy': float(selected_row['test_accuracy']), 'macro_f1': float(selected_row['macro_f1']),
    'data_zip_sha256': DATA_SHA256, 'class_names_file': 'class_names.json', 'preprocessing_file': 'preprocessing.json',
    'fallbacks': [
        {'experiment': 'B0 LS0.05', 'source_result_root': str(LS_B0_ROOT), 'reason': '같은 Loss의 작은 Backbone 대안'},
        {'experiment': 'B0 CE', 'source_result_root': str(CE_B0_ROOT), 'reason': '작고 Test 성능이 비슷한 대안'},
    ],
    'limitations': ['실제 USB 현미경 환자 데이터 미검증', '공개 Test 성능을 실제 장비 성능으로 해석할 수 없음', '원천 JSON이 없어 다중 라벨 여부 확인 불가', '사람·촬영 세션 단위 누수 확인 불가', '정상 및 범위 밖 입력 거부 기능 없음'],
}
(PACKAGE_ROOT / 'package_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
model_card = f'''# MediFlow Hair public-data candidate v1

## 상태
실제 USB 현미경 데이터로 검증하기 전의 공개 데이터 후보입니다. 의료 진단 모델로 확정된 상태가 아닙니다.

## 모델
- EfficientNet-B1, ImageNet 사전학습
- 입력: 256×256 RGB float32, 픽셀 0–255
- 모델 내부 Rescaling(1/255), 외부 정규화 금지
- Label Smoothing 0.05
- Stage 1 15 Epoch, Stage 2 총 15 Epoch
- 클래스 순서: {CLASS_NAMES}

## 공개 데이터 결과
- Validation Accuracy: {selected_row['validation_accuracy']:.6f}
- Test Accuracy: {selected_row['test_accuracy']:.6f}
- Macro F1: {selected_row['macro_f1']:.6f}

## 한계
실제 장비 검증, Domain Gap 분석, 정상 및 범위 밖 입력 처리는 완료되지 않았습니다.
'''
(PACKAGE_ROOT / 'MODEL_CARD.md').write_text(model_card, encoding='utf-8')
zip_path = Path(shutil.make_archive(str(PACKAGE_ROOT), 'zip', root_dir=PACKAGE_ROOT.parent, base_dir=PACKAGE_ROOT.name))
print('후보 패키지:', PACKAGE_ROOT)
print('후보 ZIP:', zip_path)
print('모델 SHA-256:', model_sha256)
for path in sorted(PACKAGE_ROOT.iterdir()): print(' -', path.name)


## 완료 후

마지막 셀에 출력되는 후보 ZIP을 보관하세요. 다음 필수 연구 단계는 실제 USB 현미경 검증 데이터 확보와 Domain Gap 분석입니다.
